# FlashNystrom vs the sub-quadratic field: MQAR recall

Apples-to-apples multi-query associative recall. Every mixer runs in ONE
environment: same backbone, same seeds and data, only the attention-slot operator
changes. Backends: `sdpa`, `linear_attention`, `nystrom_reference`,
`flash_nystrom`, `flash_nystrom_tc`, `hyena`, `mamba`.

**GPU requirement.** The `flash_nystrom` kernel and the `mamba-ssm` kernels need
compute capability >= 8.0 (A100 = 8.0, L4/L40S = 8.9, H100 = 9.0). On Colab's
free **T4 (7.5)** flash_nystrom is skipped and Mamba falls back to a pure-PyTorch
scan ~1000x slower; use an **A100 or L4** runtime for a full, fast run.

MQAR is bimodal (a seed either learns recall or sits on the ~48% plateau), so a
few seeds do not resolve the operators. `N_SEEDS` defaults to 5; raise it for
tighter estimates. Per-epoch loss/recall streams live below each run.

In [ ]:
import torch, subprocess, sys

def run_streaming(cmd):
    """Run cmd and stream its stdout into the notebook cell live. subprocess
    output goes to the OS stdout fd, which Colab does NOT capture; piping it and
    re-printing routes it through IPython's captured sys.stdout. `python -u` in
    the command keeps it unbuffered so lines appear as they are produced."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end=""); sys.stdout.flush()
    p.wait()
    return p.returncode

print(subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
CC = torch.cuda.get_device_capability()
print("compute capability:", CC, "| flash_nystrom + mamba-ssm kernels supported:", CC >= (8, 0))

In [ ]:
# Clone (with the CUTLASS submodule the kernel build needs), compile flash_nystrom
# for this arch, and install the real Mamba CUDA kernels.
import os, torch
%cd /content
!rm -rf FlashNystrom
!git clone --recursive -q https://github.com/athrva98/FlashNystrom.git
%cd /content/FlashNystrom
!pip -q install einops
CC = torch.cuda.get_device_capability()
if CC >= (8, 0):
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{CC[0]}.{CC[1]}"
    os.environ["FLASH_NYSTROM_LAX_BUILD"] = "1"  # tolerate 3rd-party header warnings
    !pip install -e . --no-build-isolation
    # Real Mamba kernels. Their setup.py imports torch, so pip build isolation
    # (a fresh env without torch) fails at 'getting requirements to build wheel';
    # --no-build-isolation uses the torch already installed. causal-conv1d first,
    # then mamba-ssm (which depends on it). These build from source (~5-15 min).
    !pip -q install ninja packaging
    !pip install causal-conv1d --no-build-isolation
    !pip install mamba-ssm --no-build-isolation
    import flash_nystrom
    print("flash_nystrom built, version", flash_nystrom.__version__)
    from paper.mqar.baselines import _HAS_MAMBA_CUDA
    print("Mamba CUDA kernels available:", _HAS_MAMBA_CUDA,
          "(if False, Mamba is slow -- check the mamba-ssm install log above)")
else:
    print("sm < 8.0: skipping kernel builds; pure-PyTorch backends only (Mamba slow)")

In [ ]:
import os, torch
N_SEEDS = 5
BACKENDS = ["sdpa", "linear_attention", "nystrom_reference",
            "flash_nystrom", "flash_nystrom_tc", "hyena", "mamba"]
if torch.cuda.get_device_capability() < (8, 0):
    BACKENDS = [b for b in BACKENDS if "flash_nystrom" not in b]
os.makedirs("runs/mqar_colab", exist_ok=True)
for b in BACKENDS:
    for s in range(N_SEEDS):
        out = f"runs/mqar_colab/mqar_{b}_seed{s}.json"
        if os.path.exists(out):
            print("skip", out); continue
        print(f"
===== {b} seed {s} =====", flush=True)
        run_streaming(["python", "-u", "-m", "paper.mqar.train", "--backend", b,
            "--seed", str(s), "--kappa_star", "0", "--seq_len", "256",
            "--num_kv_pairs", "16", "--num_landmarks", "64", "--newton_iter", "6",
            "--grad_clip", "1.0", "--batch_size", "256", "--epochs", "64",
            "--num_train", "20000", "--num_test", "2000", "--out_json", out])

In [ ]:
import json, glob, statistics as st
rows = {}
for f in glob.glob("runs/mqar_colab/*.json"):
    d = json.load(open(f)); rows.setdefault(d["backend"], []).append(d["best_recall"])
order = ["sdpa", "linear_attention", "nystrom_reference",
         "flash_nystrom", "flash_nystrom_tc", "hyena", "mamba"]
print("MQAR recall (seq256 / 16kv / dim128 / m64 / 64ep / kappa=0):")
print(f"{'backend':<20} {'n':>2} {'mean':>7} {'sd':>6}  seeds")
for b in order:
    if b not in rows: continue
    v = sorted(rows[b]); m = st.mean(v); sd = st.stdev(v) if len(v) > 1 else 0.0
    print(f"{b:<20} {len(v):>2} {m:7.2f} {sd:6.2f}  {['%.1f'%x for x in v]}")

## Optional: Hyena d>=N correctness gate

Zoology (Arora et al. 2023) shows gated convolutions like Hyena solve MQAR only
once model dimension `d >= N`. Hyena at `d = 256, 512` (>= N = 256) should reach
high recall; if it stays on the plateau even there, the vendored Hyena is
misconfigured rather than faithfully failing at `d < N`.

In [ ]:
import os, json
os.makedirs("runs/mqar_gate", exist_ok=True)
for d in [256, 512]:
    out = f"runs/mqar_gate/hyena_d{d}_seed0.json"
    if not os.path.exists(out):
        print(f"
===== hyena dim {d} seed 0 =====", flush=True)
        run_streaming(["python", "-u", "-m", "paper.mqar.train", "--backend", "hyena",
            "--seed", "0", "--dim", str(d), "--kappa_star", "0", "--seq_len", "256",
            "--num_kv_pairs", "16", "--grad_clip", "1.0", "--batch_size", "256",
            "--epochs", "64", "--num_train", "20000", "--num_test", "2000",
            "--out_json", out])
for d in [256, 512]:
    r = json.load(open(f"runs/mqar_gate/hyena_d{d}_seed0.json"))
    print(f"hyena d={d}: recall {r['best_recall']:.1f}%")